## Download code


In [1]:
!git clone https://github.com/mahdiNahidian/Gformer
!ls

In [1]:
import sys
if not 'Gformer' in sys.path:
    sys.path += ['Gformer']

## Experiments: Train and Test

In [2]:
from utils.tools import dotdict
from exp.exp_gformer import Exp_Gformer
import torch
import os
import pandas as pd

# e = 2 , d = 1

In [3]:
args = dotdict()

args.model = 'gformer'
args.data = 'custom'
args.root_path = '../content'
args.data_path = 'AAPL.csv'
args.checkpoints = './gformer_checkpoints'

args.features = 'MS'
args.target = 'close'
args.c_out = 1
args.freq = 'd'

df = pd.read_csv(f"{args.root_path}/{args.data_path}")
df['date'] = pd.to_datetime(df['date'])
num_features = len(df.columns) - 1

args.enc_in = num_features
args.dec_in = num_features
args.c_out = 1

args.seq_len = 96
args.label_len = 48
args.pred_len = 24

args.factor = 5
args.d_model = 512
args.n_heads = 8
args.e_layers = 2
args.d_layers = 1
args.d_ff = 2048
args.dropout = 0.05
args.attn = 'custom'
args.embed = 'timeF'
args.activation = 'gelu'
args.distil = True
args.output_attention = False
args.mix = True
args.padding = 0

args.batch_size = 32
args.learning_rate = 0.0001
args.loss = 'mse'
args.lradj = 'type1'
args.use_amp = False

args.num_workers = 0
args.itr = 1
args.train_epochs = 10
args.patience = 3
args.des = 'exp'

args.use_gpu = True if torch.cuda.is_available() else False
args.gpu = 0
args.use_multi_gpu = False
args.devices = '0,1,2,3'


In [4]:
args.use_gpu = True if torch.cuda.is_available() and args.use_gpu else False

if args.use_gpu and args.use_multi_gpu:
    args.devices = args.devices.replace(' ','')
    device_ids = args.devices.split(',')
    args.device_ids = [int(id_) for id_ in device_ids]
    args.gpu = args.device_ids[0]

In [5]:
# Set arguments by using data name
data_parser = {
    'ETTh1': {'data': 'ETTh1.csv', 'T': 'OT', 'M': [7, 7, 7], 'S': [1, 1, 1], 'MS': [7, 7, 1]},
    'ETTh2': {'data': 'ETTh2.csv', 'T': 'OT', 'M': [7, 7, 7], 'S': [1, 1, 1], 'MS': [7, 7, 1]},
    'ETTm1': {'data': 'ETTm1.csv', 'T': 'OT', 'M': [7, 7, 7], 'S': [1, 1, 1], 'MS': [7, 7, 1]},
    'ETTm2': {'data': 'ETTm2.csv', 'T': 'OT', 'M': [7, 7, 7], 'S': [1, 1, 1], 'MS': [7, 7, 1]},
    'custom': {'data': 'AAPL.csv', 'T': 'close'}
}

if args.data in data_parser.keys():
    data_info = data_parser[args.data]
    args.data_path = data_info['data']
    args.target = data_info['T']

    df = pd.read_csv(os.path.join(args.root_path, args.data_path))
    num_features = len(df.columns) - 1

    if args.features == 'M':
        args.enc_in, args.dec_in, args.c_out = [num_features, num_features, 1]
    elif args.features == 'S':
        args.enc_in, args.dec_in, args.c_out = [1, 1, 1]
    elif args.features == 'MS':
        args.enc_in, args.dec_in, args.c_out = [num_features, num_features, 1]


In [6]:
args.detail_freq = args.freq
args.freq = args.freq[-1:]

In [7]:
print('Args in experiment:')
print(args)

In [8]:
Exp = Exp_Gformer

In [9]:
import numpy as np
np.Inf = np.inf

In [15]:
for ii in range(args.itr):
    # setting record of experiments
    setting = '{}_{}_ft{}_sl{}_ll{}_pl{}_dm{}_nh{}_el{}_dl{}_df{}_at{}_fc{}_eb{}_dt{}_mx{}_{}_{}'.format(args.model, args.data, args.features,
                args.seq_len, args.label_len, args.pred_len,
                args.d_model, args.n_heads, args.e_layers, args.d_layers, args.d_ff, args.attn, args.factor, args.embed, args.distil, args.mix, args.des, ii)

    # set experiments
    exp = Exp(args)

    # train
    print('>>>>>>>start training : {}>>>>>>>>>>>>>>>>>>>>>>>>>>'.format(setting))
    exp.train(setting)

    # test
    print('>>>>>>>testing : {}<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<'.format(setting))
    exp.test(setting)

    torch.cuda.empty_cache()